In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lit, when, current_timestamp,
    datediff, initcap
)

# Read from bronze
bronze_df = spark.table("bronze.projects")
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:

cleaned_df = (
    bronze_df
    # Trim whitespace
    .withColumn("client",        trim(col("client")))
    .withColumn("practice_area", trim(col("practice_area")))
    .withColumn("project_type",  trim(col("project_type")))
    .withColumn("status",        initcap(trim(col("status"))))

    # Calculate project duration in days
    .withColumn("duration_days", datediff(col("end_date"), col("start_date")))

    # Validate date logic
    .withColumn("date_valid", when(
        col("end_date") > col("start_date"),
        lit(True)).otherwise(lit(False)))

    # Flag unusually short (< 14 days) or long (> 365 days) projects
    .withColumn("duration_flag", when(
        col("duration_days") < 14, lit("Too short"))
        .when(col("duration_days") > 365, lit("Too long"))
        .otherwise(lit("Normal")))

    # Validate headcount
    .withColumn("headcount_valid", when(
        col("headcount").between(1, 50),
        lit(True)).otherwise(lit(False)))

    # Flag nulls in critical fields
    .withColumn("has_nulls", when(
        col("project_id").isNull() |
        col("client").isNull() |
        col("start_date").isNull() |
        col("end_date").isNull(),
        lit(True)).otherwise(lit(False)))

    # Add audit columns
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.projects"))
)

# Drop duplicates on project_id
cleaned_df = cleaned_df.dropDuplicates(["project_id"])

print(f"Silver row count: {cleaned_df.count()}")
print(f"Invalid dates: {cleaned_df.filter(col('date_valid') == False).count()}")
print(f"Duration flags: {cleaned_df.filter(col('duration_flag') != 'Normal').count()}")
print(f"Invalid headcount: {cleaned_df.filter(col('headcount_valid') == False).count()}")
print(f"Rows with nulls: {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.projects")
)

print(f"Written to silver.projects: {spark.table('silver.projects').count()} rows")